题目地址：https://tianchi.aliyun.com/competition/entrance/531830/information

学习赛（第三季及以后）成绩：0.7311

软件环境：Windows11、Python 3.14.4t（自由线程版本）、numpy 2.4.4、scikit-learn 1.8.0、pandas 3.0.2、SciPy 1.17.1

硬件环境：13th Gen Core(TM) i5-13500H（12核、16线程）、16GB内存、64GB虚拟内存

每个模型训练限时：15min

In [171]:
import winsound
import pickle
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

random.seed(0)
pd.set_option('display.max_columns', None)

In [172]:
train = pd.read_csv('train.csv', parse_dates=['issueDate', 'earliesCreditLine'],
                    date_format={"earliesCreditLine": "%b-%Y"})

# 特征变量分析

## isDefault变量分析

In [173]:
(train['isDefault'] == 1).sum() / len(train['isDefault'])

np.float64(0.1995125)

# 数据清洗

In [174]:
def clear_data(df):
    df['installment'] = df['installment'] / df['annualIncome']
    df.loc[df['installment'] == np.inf, 'installment'] = pd.NA

    subGrade = [f'{char}{num}' for char in "ABCDEFG" for num in range(1, 6)]
    mapping = {}
    for i in range(len(subGrade)):
        mapping[subGrade[i]] = i + 1
    df['subGrade'] = df['subGrade'].replace(mapping)
    df['subGrade'] = df['subGrade'].astype('Int16')

    df['employmentLength'] = (
        df['employmentLength'].replace('< 1 year', '0')
        .str.extract(r'(\d+)', expand=False)
        .astype('Int16'))
    df.fillna({'employmentLength': 0}, inplace=True)

    df['issueMonth'] = df['issueDate'].dt.month

    df['livePubRec'] = df['pubRec'] - df['pubRecBankruptcies']

    df.drop(
        columns=['id', 'loanAmnt', 'grade', 'annualIncome', 'policyCode', 'pubRecBankruptcies', 'revolBal', 'totalAcc'],
        inplace=True)

    # 注意：Excel的基准通常被视为 1899-12-30
    base_date = pd.Timestamp('1899-12-30')
    # 计算天数差并转换为整数
    df['issueDate'] = (df['issueDate'] - base_date).dt.days
    df['earliesCreditLine'] = (df['earliesCreditLine'] - base_date).dt.days

    for col in df.columns:
        if df[col].dtype == 'float':
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif df[col].dtype == 'int':
            df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


train = clear_data(train)

# 数据全景

In [175]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 41 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   term                800000 non-null  int8   
 1   interestRate        800000 non-null  float32
 2   installment         799771 non-null  float32
 3   subGrade            800000 non-null  Int16  
 4   employmentTitle     799999 non-null  float32
 5   employmentLength    800000 non-null  Int16  
 6   homeOwnership       800000 non-null  int8   
 7   verificationStatus  800000 non-null  int8   
 8   issueDate           800000 non-null  int32  
 9   isDefault           800000 non-null  int8   
 10  purpose             800000 non-null  int8   
 11  postCode            799999 non-null  float32
 12  regionCode          800000 non-null  int8   
 13  dti                 799761 non-null  float32
 14  delinquency_2years  800000 non-null  float32
 15  ficoRangeLow        800000 non-null  float32


In [176]:
train

,term,interestRate,installment,subGrade,employmentTitle,employmentLength,homeOwnership,verificationStatus,issueDate,isDefault,purpose,postCode,regionCode,dti,delinquency_2years,ficoRangeLow,ficoRangeHigh,openAcc,pubRec,revolUtil,initialListStatus,applicationType,earliesCreditLine,title,n0,n1,n2,n3,n4,n5,n6,n7,n8,n9,n10,n11,n12,n13,n14,issueMonth,livePubRec
0,5,19.52,0.008345,22,320.0,2,2,2,41821,1,1,137.0,32,17.049999,0.0,730.0,734.0,7.0,0.0,48.900002,0,0,37104,1.0,0.0,2.0,2.0,2.0,4.0,9.0,8.0,4.0,12.0,2.0,7.0,0.0,0.0,0.0,2.0,7,0.0
1,5,18.49,0.010041,17,219843.0,5,0,2,41122,0,0,156.0,18,27.830000,0.0,700.0,704.0,13.0,0.0,38.900002,1,0,37377,1723.0,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,13.0,NaN,NaN,NaN,NaN,8,0.0
2,5,16.99,0.004029,18,31698.0,8,0,2,42278,0,0,337.0,14,22.770000,0.0,675.0,679.0,11.0,0.0,51.799999,0,0,38838,0.0,0.0,0.0,3.0,3.0,0.0,0.0,21.0,4.0,5.0,3.0,11.0,0.0,0.0,0.0,4.0,10,0.0
3,3,7.26,0.002889,4,46854.0,10,1,1,42217,0,4,148.0,11,17.209999,0.0,685.0,689.0,9.0,0.0,52.599998,1,0,36281,4.0,6.0,4.0,6.0,6.0,4.0,16.0,4.0,7.0,21.0,6.0,9.0,0.0,0.0,0.0,1.0,8,0.0
4,3,12.99,0.003485,12,54.0,0,1,2,42430,0,10,301.0,21,32.160000,0.0,690.0,694.0,12.0,0.0,32.000000,0,0,28338,11.0,1.0,2.0,7.0,7.0,2.0,4.0,9.0,10.0,15.0,7.0,12.0,0.0,0.0,0.0,4.0,3,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799995,3,14.49,0.011950,14,2659.0,7,1,0,42552,0,0,242.0,8,19.030001,0.0,710.0,714.0,14.0,0.0,46.400002,1,0,40756,0.0,0.0,5.0,10.0,10.0,6.0,6.0,2.0,12.0,13.0,10.0,14.0,0.0,0.0,0.0,3.0,7,0.0
799996,3,7.90,0.005373,4,29205.0,10,0,2,41365,0,4,563.0,10,15.720000,1.0,720.0,724.0,7.0,0.0,98.400002,0,0,32629,33369.0,0.0,2.0,2.0,2.0,2.0,15.0,16.0,2.0,19.0,2.0,7.0,0.0,0.0,0.0,0.0,4,0.0
799997,3,13.33,0.003125,13,2582.0,10,1,2,42278,1,0,47.0,17,12.110000,1.0,670.0,674.0,5.0,0.0,51.900002,1,0,37438,0.0,2.0,1.0,4.0,4.0,1.0,4.0,26.0,4.0,10.0,4.0,5.0,0.0,0.0,1.0,4.0,10,0.0
799998,3,6.92,0.006168,4,151.0,10,0,2,42036,0,4,34.0,18,29.250000,0.0,675.0,679.0,16.0,0.0,61.299999,1,0,34335,4.0,0.0,5.0,8.0,8.0,7.0,10.0,6.0,12.0,22.0,8.0,16.0,0.0,0.0,0.0,5.0,2,0.0


In [177]:
train.describe()

,term,interestRate,installment,subGrade,employmentTitle,employmentLength,homeOwnership,verificationStatus,issueDate,isDefault,purpose,postCode,regionCode,dti,delinquency_2years,ficoRangeLow,ficoRangeHigh,openAcc,pubRec,revolUtil,initialListStatus,applicationType,earliesCreditLine,title,n0,n1,n2,n3,n4,n5,n6,n7,n8,n9,n10,n11,n12,n13,n14,issueMonth,livePubRec
count,800000.000000,800000.000000,799771.000000,800000.0,799999.000000,800000.0,800000.000000,800000.000000,800000.000000,800000.000000,800000.000000,799999.000000,800000.000000,799761.000000,800000.000000,800000.000000,800000.000000,800000.000000,800000.000000,799469.000000,800000.000000,800000.000000,800000.000000,799999.000000,759730.000000,759730.000000,759730.000000,759730.000000,766761.000000,759730.000000,759730.000000,759730.000000,759729.000000,759730.000000,766761.000000,730248.000000,759730.000000,759730.000000,759730.000000,800000.000000,799595.000000
mean,3.482745,13.238391,0.014094,11.69151,72005.351562,5.616248,0.614213,1.009683,42161.406960,0.199513,1.745982,258.535645,16.385758,18.284557,0.318239,696.204102,700.204224,11.598020,0.214915,51.790737,0.416953,0.019267,36221.083060,1754.113525,0.511932,3.642329,5.642648,5.642648,4.735641,8.107937,8.575994,8.282953,14.622488,5.592345,11.643895,0.000815,0.003384,0.089366,2.178606,6.508888,0.080847
std,0.855832,4.765757,2.458774,6.446515,106585.640625,3.844726,0.675749,0.782716,591.067525,0.399634,2.367453,200.037445,11.036679,11.150155,0.880325,31.865993,31.866674,5.475286,0.606467,24.516125,0.493055,0.137464,2778.912974,7941.474121,1.333266,2.246825,3.302810,3.302810,2.949969,4.799210,7.400536,4.561689,8.124610,3.216184,5.484104,0.030075,0.062041,0.509069,1.844377,3.450177,0.464047
min,3.000000,5.310000,0.000006,1.0,0.000000,0.0,0.000000,0.000000,39234.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,630.000000,634.000000,0.000000,0.000000,0.000000,0.000000,0.000000,16072.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,3.000000,9.750000,0.003858,7.0,427.000000,2.0,0.000000,0.000000,41821.000000,0.000000,0.000000,103.000000,8.000000,11.790000,0.000000,670.000000,674.000000,8.000000,0.000000,33.400002,0.000000,0.000000,34790.000000,0.000000,0.000000,2.000000,3.000000,3.000000,3.000000,5.000000,4.000000,5.000000,9.000000,3.000000,8.000000,0.000000,0.000000,0.000000,1.000000,3.000000,0.000000
50%,3.000000,12.740000,0.006020,11.0,7755.000000,6.0,1.000000,1.000000,42217.000000,0.000000,0.000000,203.000000,14.000000,17.610001,0.000000,690.000000,694.000000,11.000000,0.000000,52.099998,0.000000,0.000000,36739.000000,1.000000,0.000000,3.000000,5.000000,5.000000,4.000000,7.000000,7.000000,7.000000,13.000000,5.000000,11.000000,0.000000,0.000000,0.000000,2.000000,7.000000,0.000000
75%,3.000000,15.990000,0.008789,15.0,117663.500000,10.0,1.000000,2.000000,42552.000000,0.000000,4.000000,395.000000,22.000000,24.059999,0.000000,710.000000,714.000000,14.000000,0.000000,70.699997,1.000000,0.000000,38139.000000,5.000000,0.000000,5.000000,7.000000,7.000000,6.000000,11.000000,11.000000,10.000000,19.000000,7.000000,14.000000,0.000000,0.000000,0.000000,3.000000,10.000000,0.000000
max,5.000000,30.990000,1100.660034,35.0,378351.000000,10.0,5.000000,2.000000,43435.000000,1.000000,13.000000,940.000000,50.000000,999.000000,39.000000,845.000000,850.000000,86.000000,86.000000,892.299988,1.000000,1.000000,42278.000000,61680.000000,51.000000,33.000000,63.000000,63.000000,49.000000,70.000000,132.000000,79.000000,128.000000,45.000000,82.000000,4.000000,4.000000,39.000000,30.000000,12.000000,85.000000


# 创建变量预处理器

In [178]:
X_train = train.drop(columns=['isDefault'])
y_train = train['isDefault']

numeric_features = ['term', 'interestRate', 'installment', 'subGrade',
                    'employmentLength', 'issueDate',
                    'dti', 'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh',
                    'openAcc', 'pubRec', 'livePubRec',
                    'revolUtil', 'earliesCreditLine', 'n0', 'n1', 'n2', 'n3', 'n4', 'n5',
                    'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14']
categorical_features = ['employmentTitle', 'homeOwnership', 'verificationStatus',
                        'purpose', 'postCode', 'regionCode', 'initialListStatus',
                        'applicationType', 'title', 'issueMonth']

# processor1
# 数值型：填充中位数、标准化
# 分类型：目标编码、标准化
num_median_standar_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_target_transformer = Pipeline(steps=[
    ("target", TargetEncoder(target_type='binary', random_state=0)),
    ("scaler", StandardScaler())
])
processor1 = ColumnTransformer(
    transformers=[
        ('num', num_median_standar_transformer, numeric_features),
        ('cat', cat_target_transformer, categorical_features)
    ]
)

# processor2
# 数值型：填充中位数
# 分类型：目标编码
cat_target_transformer = Pipeline(steps=[
    ("target", TargetEncoder(target_type='binary', random_state=0))
])
processor2 = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy="median"), numeric_features),
        ('cat', cat_target_transformer, categorical_features)
    ],
).set_output(transform='pandas')


# 结果记录函数

In [179]:
results = pd.DataFrame({
    '最佳估计器': [],
    '最佳估计器实例': [],
    '最佳估计器的平均交叉验证分数': [],
    '训练耗时': []
})


def record(model, random_search, start_time, end_time):
    global results

    result = {
        '最佳估计器': [model],
        '最佳估计器实例': [random_search.best_estimator_],
        '最佳估计器的平均交叉验证分数': [random_search.best_score_],
        '训练耗时': [end_time - start_time]
    }

    print(f'最佳估计器的平均交叉验证分数:{result['最佳估计器的平均交叉验证分数'][0]:.4f}')
    print(f'训练耗时:{result['训练耗时'][0] / 60:.2f}min')

    temp = pd.DataFrame(result)
    results = pd.concat([results, temp], axis=0)

    with open(f'{model}.pkl', 'wb') as f:
        pickle.dump(random_search.best_estimator_, f)
        print(f"已保存{model}")

    # 完成声音提醒
    winsound.PlaySound("SystemAsterisk", winsound.SND_ALIAS)



# 模型训练

## DecisionTreeClassifier

In [180]:
file_path = Path('DecisionTreeClassifier.pkl')
if not file_path.exists():
    start_time = time.time()

    decision_tree = make_pipeline(processor2,
                                  DecisionTreeClassifier(random_state=0, class_weight="balanced"))

    param_dist = [
        #默认参数
        #{},
        {
            # 树深度：控制模型复杂度，80万数据可以允许较深的树
            'decisiontreeclassifier__max_depth': randint(8, 40),

            # 最小分割样本数：防止过拟合的关键参数
            'decisiontreeclassifier__min_samples_split': randint(10, 500),

            # 叶子节点最小样本数：与min_samples_split配合
            'decisiontreeclassifier__min_samples_leaf': randint(5, 200),

            # 最大特征数：sqrt和log2是常用选择，也可以用具体数值
            'decisiontreeclassifier__max_features': ['sqrt', 'log2', None],

            # 分割标准
            'decisiontreeclassifier__criterion': ['gini', 'entropy'],

            # 分割策略：random可以加速训练且有时效果更好
            'decisiontreeclassifier__splitter': ['best', 'random'],
        }
    ]

    random_search = RandomizedSearchCV(
        decision_tree,
        param_dist,
        scoring='roc_auc',
        n_iter=10,
        cv=3,
        random_state=0,
        n_jobs=9
    )
    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('DecisionTreeClassifier', random_search, start_time, end_time)


默认参数：

最佳估计器的平均交叉验证分数:0.5554

训练耗时:0.91min


最佳参数：

最佳估计器的平均交叉验证分数:0.7070

训练耗时:1.36min

## LogisticRegression

In [181]:
file_path = Path('LogisticRegression.pkl')
if not file_path.exists():
    start_time = time.time()

    clf = make_pipeline(processor1, LogisticRegression(class_weight="balanced", max_iter=200, random_state=0))

    param_dist = [
        # 默认参数
        #{},
        # saga + L1 正则化（可做特征选择）
        {
            "logisticregression__solver": ["saga"],
            "logisticregression__l1_ratio": [1.0],
            "logisticregression__C": loguniform(1e-3, 1e2),
            "logisticregression__max_iter": [200, 300],
        },
        # saga + L2 正则化
        {
            "logisticregression__solver": ["saga"],
            "logisticregression__l1_ratio": [0.0],
            "logisticregression__C": loguniform(1e-3, 1e2),
            "logisticregression__max_iter": [100, 200],
        },
        # saga + ElasticNet 混合正则化
        {
            "logisticregression__solver": ["saga"],
            "logisticregression__l1_ratio": [0.2, 0.5, 0.8],
            "logisticregression__C": loguniform(1e-3, 1e2),
            "logisticregression__max_iter": [200, 300],
        },
        # lbfgs + L2（收敛最快，仅支持 l1_ratio=0.0）
        {
            "logisticregression__solver": ["lbfgs"],
            "logisticregression__l1_ratio": [0.0],
            "logisticregression__C": loguniform(1e-3, 1e2),
            "logisticregression__max_iter": [100, 200],
        },
    ]

    random_search = RandomizedSearchCV(
        clf,
        param_dist,
        scoring='roc_auc',
        n_iter=15,
        cv=3,
        n_jobs=8
    )
    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('LogisticRegression', random_search, start_time, end_time)


默认参数：

最佳估计器的平均交叉验证分数:0.7170

训练耗时:0.42min


最佳参数：

最佳估计器的平均交叉验证分数:0.7167

训练耗时:10.05min

## SVC

In [182]:
file_path = Path('SVC.pkl')
if not file_path.exists():
    start_time = time.time()

    svc = make_pipeline(processor1,
                        SVC(class_weight='balanced', probability=True, random_state=0))

    param_grid = [
        # 默认参数
        {},
    ]

    random_search = RandomizedSearchCV(
        svc,
        param_grid,
        scoring='roc_auc',
        cv=5,
        n_iter=1,
        random_state=0,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('SVC', random_search, start_time, end_time)

# AI: SVC的时间复杂度是 O(n²) ~ O(n³)，80万样本直接训练SVC几乎不可能在15分钟内完成

默认参数：

最佳估计器的平均交叉验证分数:0.7170

训练耗时:0.46min


最佳参数：

最佳估计器的平均交叉验证分数:

训练耗时:

## LinearSVC

In [183]:
file_path = Path('LinearSVC.pkl')
if not file_path.exists():
    start_time = time.time()

    linear_svc = make_pipeline(processor1, LinearSVC(class_weight='balanced', random_state=0))

    param_grid = [
        #默认参数
        #{},
        {
            # C 是 LinearSVC 最重要的参数，使用对数均匀分布覆盖多个数量级
            'linearsvc__C': loguniform(1e-4, 1e3),

            # 必须设为 False，否则 80 万样本会导致极慢的训练速度
            'linearsvc__dual': [False],

            # 容忍度：太小会显著增加训练时间，太大影响精度
            # 1e-3 到 1e-2 是合理范围
            'linearsvc__tol': loguniform(1e-4, 1e-1),

            # 最大迭代次数：确保模型能收敛
            # LinearSVC 在 C 较大时可能需要更多迭代
            'linearsvc__max_iter': [5000, 10000],
        }
    ]
    random_search = RandomizedSearchCV(
        linear_svc,
        param_grid,
        scoring='roc_auc',
        cv=3,
        n_iter=10,
        random_state=0,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('LinearSVC', random_search, start_time, end_time)


默认参数：

最佳估计器的平均交叉验证分数:0.7169

训练耗时:0.72min


最佳参数：

最佳估计器的平均交叉验证分数:0.7166

训练耗时:6.32min

## KNeighborsClassifier

In [184]:
file_path = Path('KNeighborsClassifier.pkl')
# 太耗时、准确率不高，放弃
#if not file_path.exists():
if False:
    start_time = time.time()

    knn = make_pipeline(processor1, KNeighborsClassifier(n_jobs=-1))

    param_grid = [
        #默认参数
        {},
        # {
        #     # 核心参数：邻居数
        #     # 范围5-150，使用均匀分布（比loguniform更适合KNN）
        #     'kneighborsclassifier__n_neighbors': randint(5, 146),  # randint(low, high)生成[low, high)的整数
        #
        #     # 权重策略
        #     'kneighborsclassifier__weights': ['uniform', 'distance'],
        #
        #     # 距离度量
        #     # minkowski是通用选择，p=1是曼哈顿，p=2是欧氏
        #     'kneighborsclassifier__metric': ['minkowski', 'euclidean', 'manhattan'],
        #     'kneighborsclassifier__metric_params': [None],  # 配合minkowski使用
        #
        #     # Minkowski距离的p参数（仅当metric='minkowski'时有效）
        #     'kneighborsclassifier__p': [1, 2],
        #
        #     # 叶节点大小（使用ball_tree/kd_tree时重要）
        #     # 默认30，可调范围10-50
        #     'kneighborsclassifier__leaf_size': randint(10, 41),
        # }
    ]

    random_search = RandomizedSearchCV(
        knn,
        param_grid,
        scoring='roc_auc',
        cv=5,
        n_iter=1,
        random_state=0,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('KNeighborsClassifier', random_search, start_time, end_time)

# 耗时

默认参数：

最佳估计器的平均交叉验证分数:0.6227

训练耗时:8.71min


最佳参数：

最佳估计器的平均交叉验证分数:

训练耗时:

## 集成学习Bagging

### RandomForestClassifier

In [185]:
file_path = Path('RandomForestClassifier.pkl')
if not file_path.exists():
    start_time = time.time()

    random_forest = make_pipeline(processor2,
                                  RandomForestClassifier(random_state=0, class_weight="balanced", n_jobs=-1))

    param_dist = [
        #默认参数
        #{},
        {
            # 树的数量：大幅减少到30-60棵
            'randomforestclassifier__n_estimators': randint(30, 60),

            # 最大深度：更严格限制在8-15
            'randomforestclassifier__max_depth': randint(8, 15),

            # 分裂最小样本数：大幅增加到50-200
            'randomforestclassifier__min_samples_split': randint(50, 200),

            # 叶节点最小样本数：增加到20-100
            'randomforestclassifier__min_samples_leaf': randint(20, 100),

            # 最大特征数：使用log2或更小的比例
            'randomforestclassifier__max_features': ['log2', 0.2, 0.3],

            # 采样比例：只使用30%-50%的数据训练每棵树
            # 这是加速训练的最有效方法之一
            'randomforestclassifier__max_samples': [0.3, 0.4, 0.5],

        }
    ]

    random_search = RandomizedSearchCV(
        random_forest,
        param_dist,
        scoring='roc_auc',
        n_iter=7,
        cv=3,
        random_state=0,
        n_jobs=1
    )
    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('RandomForestClassifier', random_search, start_time, end_time)




默认参数：

最佳估计器的平均交叉验证分数:0.7178

训练耗时:2.98min


最佳参数：

最佳估计器的平均交叉验证分数:0.7243

训练耗时:2.50min


## 集成学习Boosting

### HistGradientBoostingClassifier

In [186]:
file_path = Path('HistGradientBoostingClassifier.pkl')
if not file_path.exists():
    start_time = time.time()

    HGBT = make_pipeline(
        processor1,
        HistGradientBoostingClassifier(
            class_weight="balanced",
            n_iter_no_change=15,
            tol=1e-4,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=0
        )
    )
    HGBT.set_output(transform="pandas")
    param_dist = [
        #默认参数
        {},
        # {
        #     # 核心参数：控制模型容量
        #     'histgradientboostingclassifier__max_iter': randint(200, 500),
        #     'histgradientboostingclassifier__learning_rate': loguniform(0.02, 0.15),
        #     'histgradientboostingclassifier__max_depth': [3, 5, 7, None],
        #
        #     # 正则化参数：防止过拟合
        #     'histgradientboostingclassifier__min_samples_leaf': randint(20, 100),
        #     'histgradientboostingclassifier__l2_regularization': loguniform(0.01, 10.0),
        #     'histgradientboostingclassifier__max_bins': [127, 255],
        #
        #     # 采样参数：加速训练的关键
        #     'histgradientboostingclassifier__max_leaf_nodes': [31, 63, 127],
        #     'histgradientboostingclassifier__early_stopping': [True],
        #     'histgradientboostingclassifier__validation_fraction': [0.1],
        #     'histgradientboostingclassifier__n_iter_no_change': [15],
        # }
    ]

    random_search = RandomizedSearchCV(
        HGBT,
        param_dist,
        scoring='roc_auc',
        n_iter=1,
        cv=5,
        random_state=0,
        n_jobs=10
    )
    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('HistGradientBoostingClassifier', random_search, start_time, end_time)



默认参数：

最佳估计器的平均交叉验证分数:0.7331

训练耗时:2.07min


最佳参数：

最佳估计器的平均交叉验证分数:0.7330

训练耗时:4.22min


### AdaBoostClassifier

In [187]:
file_path = Path('AdaBoostClassifier.pkl')
if not file_path.exists():
    start_time = time.time()

    base_est = DecisionTreeClassifier(max_depth=1, class_weight="balanced", random_state=0)
    ada = make_pipeline(processor2, AdaBoostClassifier(estimator=base_est, random_state=0))

    param_grid = [
        #默认参数
        #{},
        {
            'adaboostclassifier__n_estimators': randint(50, 101),
            'adaboostclassifier__learning_rate': loguniform(0.01, 1.0),
            'adaboostclassifier__estimator__max_depth': randint(1, 4),
            'adaboostclassifier__estimator__min_samples_split': randint(2, 19),
            'adaboostclassifier__estimator__min_samples_leaf': randint(1, 10),
        }
    ]

    random_search = RandomizedSearchCV(
        ada,
        param_grid,
        scoring='roc_auc',
        n_iter=10,
        cv=3,
        random_state=0,
        n_jobs=-1,
    )

    random_search.fit(X_train, y_train)

    end_time = time.time()
    record('AdaBoostClassifier', random_search, start_time, end_time)

默认参数：

最佳估计器的平均交叉验证分数:0.7165

训练耗时:8.97min


最佳参数：

最佳估计器的平均交叉验证分数:0.7165

训练耗时:12.47min


# 特征重要性分析

In [188]:
# 太耗时，放弃
# with open('HistGradientBoostingClassifier.pkl', 'rb') as f:
#     best_model = pickle.load(f)
#     result = permutation_importance(
#         best_model,
#         X_train,
#         y_train,
#         scoring='roc_auc',
#         n_repeats=1,
#         n_jobs=-1,
#         random_state=0
#     )
#     importance_df = pd.DataFrame({
#         'feature': X_train.columns,
#         'importance_mean': result.importances_mean,
#         'importance_std': result.importances_std
#     }).sort_values(by='importance_mean', ascending=False)
#     print(importance_df)

# 生成答案

In [189]:
with open('HistGradientBoostingClassifier.pkl', 'rb') as f:
    best_model = pickle.load(f)

    test = pd.read_csv('testA.csv', parse_dates=['issueDate', 'earliesCreditLine'],
                       date_format={"earliesCreditLine": "%b-%Y"})
    test = clear_data(test)

    probabilities = best_model.predict_proba(test)

    # print("Classes order:", best_model.classes_)
    answers = pd.DataFrame({
        'id': range(800000, 800000 + len(test)),
        'isDefault': probabilities[:, 1],
    })
    answers.to_csv('answer.csv', index=False)

